# Arbitrage & Value Betting

**Chapter 9: From Predictions to Profit: Navigating the World of Soccer Betting**

## Learning Objectives

- Understand arbitrage betting and when it exists
- Calculate optimal stakes for arbitrage opportunities
- Identify value bets systematically
- Calculate expected value for betting opportunities
- Understand the reality and challenges of arbitrage
- Compare arbitrage vs value betting strategies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("Arbitrage & Value Betting Toolkit Ready!")

## Introduction

**Arbitrage betting** and **value betting** are two fundamentally different approaches to profitable betting:

### Arbitrage Betting
- **Guaranteed profit** regardless of outcome
- Exploits differences in odds across bookmakers
- Rare and quickly disappearing opportunities
- No prediction required

### Value Betting
- **Positive expected value** over many bets
- Requires accurate probability predictions
- More common but requires skill
- Profits emerge over the long run

Let's explore both approaches in detail!

---

# 1. Arbitrage Betting

**Arbitrage** occurs when differences in odds across bookmakers allow you to bet on all outcomes and guarantee a profit.

## 1.1 When Does Arbitrage Exist?

For a three-outcome event (1X2), arbitrage exists if:

```
1/Odds₁ + 1/OddsX + 1/Odds₂ < 1
```

Where you use the **best odds** available for each outcome across all bookmakers.

In [ ]:
def check_arbitrage(odds_home, odds_draw, odds_away):
    """
    Check if arbitrage opportunity exists.
    
    Returns:
        exists: Boolean indicating if arbitrage exists
        total_prob: Sum of implied probabilities
        profit_pct: Expected profit percentage
    """
    total_prob = (1/odds_home) + (1/odds_draw) + (1/odds_away)
    exists = total_prob < 1.0
    profit_pct = (1 / total_prob - 1) * 100 if exists else 0
    
    return exists, total_prob, profit_pct

# Example: Check for arbitrage
odds_home = 2.10
odds_draw = 3.80
odds_away = 4.20

exists, total_prob, profit_pct = check_arbitrage(odds_home, odds_draw, odds_away)

print(f"Odds: Home {odds_home}, Draw {odds_draw}, Away {odds_away}")
print(f"\nImplied Probabilities:")
print(f"  Home: {1/odds_home:.1%}")
print(f"  Draw: {1/odds_draw:.1%}")
print(f"  Away: {1/odds_away:.1%}")
print(f"  Total: {total_prob:.1%}")
print(f"\n{'✅ ARBITRAGE EXISTS!' if exists else '❌ No arbitrage'}")

if exists:
    print(f"Expected Profit: {profit_pct:.2f}%")

## 1.2 Example: Finding Arbitrage Across Bookmakers

Let's simulate a real scenario with two bookmakers offering different odds.

In [ ]:
# Create odds from two bookmakers
bookmaker_odds = pd.DataFrame({
    'Outcome': ['Home Win', 'Draw', 'Away Win'],
    'Bookmaker A': [2.10, 3.60, 4.20],
    'Bookmaker B': [1.90, 3.80, 4.00]
})

print("Odds from Two Bookmakers:\n")
print(bookmaker_odds.to_string(index=False))

# Find best odds for each outcome
best_odds = {
    'Home Win': max(bookmaker_odds.loc[0, 'Bookmaker A'], bookmaker_odds.loc[0, 'Bookmaker B']),
    'Draw': max(bookmaker_odds.loc[1, 'Bookmaker A'], bookmaker_odds.loc[1, 'Bookmaker B']),
    'Away Win': max(bookmaker_odds.loc[2, 'Bookmaker A'], bookmaker_odds.loc[2, 'Bookmaker B'])
}

print("\nBest Odds for Each Outcome:")
for outcome, odds in best_odds.items():
    bookmaker = 'A' if odds in bookmaker_odds['Bookmaker A'].values else 'B'
    print(f"  {outcome}: {odds:.2f} (Bookmaker {bookmaker})")

# Check for arbitrage using best odds
exists, total_prob, profit_pct = check_arbitrage(
    best_odds['Home Win'],
    best_odds['Draw'],
    best_odds['Away Win']
)

print(f"\nArbitrage Check:")
print(f"  Total Implied Probability: {total_prob:.1%}")
print(f"  {'✅ ARBITRAGE EXISTS!' if exists else '❌ No arbitrage'}")
if exists:
    print(f"  Expected Profit: {profit_pct:.2f}%")

## 1.3 Calculating Optimal Stakes

Once arbitrage is found, we need to calculate how much to bet on each outcome to guarantee equal profit.

In [ ]:
def calculate_arbitrage_stakes(odds_home, odds_draw, odds_away, total_stake=100):
    """
    Calculate optimal stakes for arbitrage betting.
    
    Args:
        odds_home, odds_draw, odds_away: Best odds for each outcome
        total_stake: Total amount to invest
    
    Returns:
        stakes: Dictionary with stake for each outcome
        payouts: Dictionary with payout for each outcome
        profit: Guaranteed profit
    """
    # Calculate implied probabilities
    prob_home = 1 / odds_home
    prob_draw = 1 / odds_draw
    prob_away = 1 / odds_away
    total_prob = prob_home + prob_draw + prob_away
    
    # Check if arbitrage exists
    if total_prob >= 1.0:
        return None, None, None
    
    # Calculate stakes proportional to implied probabilities
    stake_home = (prob_home / total_prob) * total_stake
    stake_draw = (prob_draw / total_prob) * total_stake
    stake_away = (prob_away / total_prob) * total_stake
    
    # Calculate payouts
    payout_home = stake_home * odds_home
    payout_draw = stake_draw * odds_draw
    payout_away = stake_away * odds_away
    
    # Calculate profit (should be same for all outcomes)
    profit = payout_home - total_stake
    
    stakes = {
        'Home Win': stake_home,
        'Draw': stake_draw,
        'Away Win': stake_away
    }
    
    payouts = {
        'Home Win': payout_home,
        'Draw': payout_draw,
        'Away Win': payout_away
    }
    
    return stakes, payouts, profit

# Calculate stakes for our example
total_stake = 100
stakes, payouts, profit = calculate_arbitrage_stakes(
    best_odds['Home Win'],
    best_odds['Draw'],
    best_odds['Away Win'],
    total_stake
)

if stakes:
    print(f"Arbitrage Betting Plan (Total Stake: ${total_stake})\n")
    print(f"{'Outcome':<12} {'Odds':<8} {'Stake':<12} {'Payout':<12} {'Profit'}")
    print("="*60)
    
    for outcome in ['Home Win', 'Draw', 'Away Win']:
        odds = best_odds[outcome]
        stake = stakes[outcome]
        payout = payouts[outcome]
        profit_val = payout - total_stake
        
        print(f"{outcome:<12} {odds:<8.2f} ${stake:<11.2f} ${payout:<11.2f} ${profit_val:.2f}")
    
    print("\n✅ Guaranteed Profit: ${:.2f} ({:.2f}%)".format(profit, (profit/total_stake)*100))
else:
    print("❌ No arbitrage opportunity exists with these odds.")

## 1.4 Visualizing Arbitrage Opportunity

In [ ]:
if stakes:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left panel: Stakes distribution
    outcomes = list(stakes.keys())
    stake_values = list(stakes.values())
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    ax1.pie(stake_values, labels=outcomes, autopct='%1.1f%%', colors=colors, startangle=90)
    ax1.set_title(f'Stake Distribution (Total: ${total_stake})', fontsize=14, fontweight='bold')
    
    # Right panel: Payouts comparison
    x = np.arange(len(outcomes))
    payout_values = list(payouts.values())
    
    bars = ax2.bar(x, payout_values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    ax2.axhline(total_stake, color='red', linestyle='--', linewidth=2, label=f'Total Stake (${total_stake})')
    ax2.axhline(total_stake + profit, color='green', linestyle='--', linewidth=2, 
                label=f'Payout (${total_stake + profit:.2f})')
    
    ax2.set_xticks(x)
    ax2.set_xticklabels(outcomes)
    ax2.set_ylabel('Amount ($)', fontsize=12)
    ax2.set_title('Guaranteed Payouts', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, payout_values)):
        ax2.text(bar.get_x() + bar.get_width()/2, val + 1, f'${val:.2f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("Notice: All three payouts are equal - guaranteed profit!")

## 1.5 The Reality of Arbitrage Betting

While mathematically elegant, arbitrage betting faces significant practical challenges:

### Challenges

**1. Rarity**
- Modern markets are highly efficient
- Opportunities disappear in seconds
- Professional algorithms compete for them

**2. Bookmaker Restrictions**
- Accounts may be limited or closed
- Maximum stakes reduced
- Bet acceptance delayed

**3. Execution Risk**
- Need simultaneous bets across bookmakers
- Odds can change before all bets are placed
- Website delays or downtime

**4. Transaction Costs**
- Deposit/withdrawal fees
- Currency conversion costs
- Tax implications

### Conclusion

Arbitrage is more of a **theoretical curiosity** than a practical long-term strategy for most bettors.

---

# 2. Value Betting

**Value betting** is the foundation of profitable betting. Unlike arbitrage, it requires skill but offers more opportunities.

## 2.1 What is a Value Bet?

A value bet occurs when:

```
Your Estimated Probability > Implied Probability from Odds
```

### Expected Value (EV)

```
EV = (Win Probability × Profit) - (Loss Probability × Stake)
```

Positive EV = Value bet!

In [ ]:
def calculate_expected_value(your_probability, odds, stake=1):
    """
    Calculate expected value of a bet.
    
    Args:
        your_probability: Your estimated win probability (0-1)
        odds: Decimal odds
        stake: Bet amount (default 1)
    
    Returns:
        ev: Expected value
        ev_pct: EV as percentage of stake
        is_value: Boolean indicating if it's a value bet
    """
    implied_prob = 1 / odds
    profit_if_win = (odds - 1) * stake
    loss_if_lose = stake
    
    ev = (your_probability * profit_if_win) - ((1 - your_probability) * loss_if_lose)
    ev_pct = (ev / stake) * 100
    is_value = ev > 0
    
    return ev, ev_pct, is_value, implied_prob

# Example: Evaluate a potential bet
your_prob = 0.55  # You think there's 55% chance of home win
odds = 2.00        # Bookmaker offers 2.00 odds (50% implied)
stake = 10

ev, ev_pct, is_value, implied_prob = calculate_expected_value(your_prob, odds, stake)

print(f"Value Bet Analysis:\n")
print(f"Your Probability: {your_prob:.1%}")
print(f"Bookmaker Odds: {odds:.2f}")
print(f"Implied Probability: {implied_prob:.1%}")
print(f"\nYour Edge: {(your_prob - implied_prob):+.1%}")
print(f"Expected Value: ${ev:+.2f}")
print(f"EV as % of Stake: {ev_pct:+.1f}%")
print(f"\n{'✅ VALUE BET!' if is_value else '❌ No value'}")

if is_value:
    print(f"\nInterpretation: Over many similar bets, you expect to make ${ev:.2f} per ${stake} wagered.")

## 2.2 Scanning for Value Bets

Let's create a function to scan multiple matches for value betting opportunities.

In [ ]:
# Generate sample matches with odds and your predictions
np.random.seed(42)

matches = []
for i in range(10):
    # Simulate bookmaker odds
    home_odds = np.random.uniform(1.5, 4.0)
    draw_odds = np.random.uniform(3.0, 4.5)
    away_odds = np.random.uniform(1.5, 5.0)
    
    # Normalize to add overround
    total = (1/home_odds + 1/draw_odds + 1/away_odds)
    overround = 1.05  # 5% margin
    home_odds = home_odds * (total / overround)
    draw_odds = draw_odds * (total / overround)
    away_odds = away_odds * (total / overround)
    
    # Your predictions (sometimes better, sometimes worse than bookmaker)
    your_home_prob = np.random.uniform(0.2, 0.6)
    your_draw_prob = np.random.uniform(0.15, 0.35)
    your_away_prob = 1 - your_home_prob - your_draw_prob
    
    matches.append({
        'match_id': i+1,
        'home_team': f'Team {chr(65+i)}',
        'away_team': f'Team {chr(75+i)}',
        'home_odds': home_odds,
        'draw_odds': draw_odds,
        'away_odds': away_odds,
        'your_home_prob': your_home_prob,
        'your_draw_prob': your_draw_prob,
        'your_away_prob': your_away_prob
    })

matches_df = pd.DataFrame(matches)

print(f"Generated {len(matches_df)} matches for analysis")
matches_df.head()

In [ ]:
def find_value_bets(matches_df, min_edge=0.05):
    """
    Scan matches for value betting opportunities.
    
    Args:
        matches_df: DataFrame with matches, odds, and predictions
        min_edge: Minimum edge required to consider it a value bet
    
    Returns:
        value_bets: DataFrame with value betting opportunities
    """
    value_bets = []
    
    for _, match in matches_df.iterrows():
        # Check each outcome
        for outcome in ['home', 'draw', 'away']:
            odds = match[f'{outcome}_odds']
            your_prob = match[f'your_{outcome}_prob']
            implied_prob = 1 / odds
            edge = your_prob - implied_prob
            
            if edge > min_edge:
                ev, ev_pct, _, _ = calculate_expected_value(your_prob, odds, stake=10)
                
                value_bets.append({
                    'match_id': match['match_id'],
                    'match': f"{match['home_team']} vs {match['away_team']}",
                    'outcome': outcome.capitalize(),
                    'odds': odds,
                    'implied_prob': implied_prob,
                    'your_prob': your_prob,
                    'edge': edge,
                    'ev_pct': ev_pct
                })
    
    return pd.DataFrame(value_bets).sort_values('edge', ascending=False)

# Find value bets
value_bets = find_value_bets(matches_df, min_edge=0.05)

print(f"Found {len(value_bets)} value betting opportunities:\n")
if len(value_bets) > 0:
    print(value_bets.to_string(index=False))
else:
    print("No value bets found with current criteria.")

## 2.3 Visualizing Value Bets

In [ ]:
if len(value_bets) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left panel: Edge distribution
    top_bets = value_bets.head(8)
    
    y_pos = np.arange(len(top_bets))
    colors = ['green' if e > 0.1 else 'orange' for e in top_bets['edge']]
    
    ax1.barh(y_pos, top_bets['edge'] * 100, color=colors, alpha=0.7, edgecolor='black')
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels([f"Match {m}" for m in top_bets['match_id']])
    ax1.set_xlabel('Edge (%)', fontsize=12)
    ax1.set_title('Top Value Bets by Edge', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Right panel: Your probability vs Implied probability
    ax2.scatter(top_bets['implied_prob'] * 100, top_bets['your_prob'] * 100, 
                s=top_bets['edge'] * 1000, alpha=0.6, c=top_bets['edge'], 
                cmap='RdYlGn', edgecolors='black', linewidth=1.5)
    
    # Add diagonal line (no value)
    lims = [0, 100]
    ax2.plot(lims, lims, 'k--', alpha=0.5, linewidth=2, label='No Value Line')
    
    ax2.set_xlabel('Implied Probability (%)', fontsize=12)
    ax2.set_ylabel('Your Probability (%)', fontsize=12)
    ax2.set_title('Value Bets: Your Prob vs Market', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap='RdYlGn', 
                                norm=plt.Normalize(vmin=top_bets['edge'].min(), 
                                                  vmax=top_bets['edge'].max()))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax2)
    cbar.set_label('Edge', fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    print("Points above the diagonal line represent value bets!")
    print("Larger circles = bigger edge")

## 2.4 Long-Term Value: The Law of Large Numbers

Value betting relies on the **law of large numbers** - over many bets, your edge materializes into profit.

In [ ]:
def simulate_value_betting(num_bets, your_prob, odds, stake=10, num_simulations=1000):
    """
    Simulate value betting over many bets.
    """
    results = []
    
    for _ in range(num_simulations):
        bankroll = 0
        for _ in range(num_bets):
            # Simulate bet outcome
            win = np.random.random() < your_prob
            if win:
                bankroll += (odds - 1) * stake
            else:
                bankroll -= stake
        
        results.append(bankroll)
    
    return np.array(results)

# Simulate value betting
your_prob = 0.55
odds = 2.00
stake = 10
num_bets_range = [10, 50, 100, 500, 1000]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for num_bets in num_bets_range:
    results = simulate_value_betting(num_bets, your_prob, odds, stake, num_simulations=1000)
    
    # Plot distribution
    ax1.hist(results, bins=50, alpha=0.5, label=f'{num_bets} bets', density=True)

ax1.axvline(0, color='red', linestyle='--', linewidth=2, label='Break-even')
ax1.set_xlabel('Final Profit/Loss ($)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Profit Distribution (1000 simulations each)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Expected profit vs number of bets
num_bets_range_detailed = range(10, 1001, 10)
expected_profits = []
std_profits = []

for num_bets in num_bets_range_detailed:
    results = simulate_value_betting(num_bets, your_prob, odds, stake, num_simulations=500)
    expected_profits.append(results.mean())
    std_profits.append(results.std())

expected_profits = np.array(expected_profits)
std_profits = np.array(std_profits)

ax2.plot(num_bets_range_detailed, expected_profits, linewidth=2, color='green', label='Expected Profit')
ax2.fill_between(num_bets_range_detailed, 
                  expected_profits - std_profits, 
                  expected_profits + std_profits, 
                  alpha=0.3, color='green', label='±1 Std Dev')
ax2.axhline(0, color='red', linestyle='--', alpha=0.5)
ax2.set_xlabel('Number of Bets', fontsize=12)
ax2.set_ylabel('Expected Profit ($)', fontsize=12)
ax2.set_title('Long-Term Convergence to Expected Value', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Insight: As number of bets increases, profit converges to expected value!")
print(f"Expected profit per bet: ${(your_prob * (odds-1) - (1-your_prob)) * stake:.2f}")

## Summary

In this notebook, you learned:

✅ **Arbitrage betting** - Guaranteed profit by exploiting odds differences

✅ **Calculating arbitrage stakes** - Optimal allocation for equal profit

✅ **Reality of arbitrage** - Practical challenges and limitations

✅ **Value betting** - Finding positive expected value opportunities

✅ **Expected value calculation** - Quantifying betting opportunities

✅ **Scanning for value** - Systematic approach to finding value bets

✅ **Long-term convergence** - Law of large numbers in action

### Key Takeaways

**1. Arbitrage vs Value**
- Arbitrage: Guaranteed profit, rare, no skill needed
- Value: Probabilistic profit, common, requires skill

**2. Expected Value is King**
- Positive EV is the only sustainable edge
- Short-term variance is inevitable
- Long-term convergence to EV

**3. Edge Matters**
- Even small edges compound over many bets
- 5% edge can be very profitable
- Consistency is more important than big wins

**4. Practical Considerations**
- Arbitrage faces execution challenges
- Value betting requires accurate predictions
- Both require disciplined bankroll management

## Practice Exercises

1. **Find arbitrage**: Check if arbitrage exists with odds: Home 2.20, Draw 3.50, Away 3.80

2. **Calculate stakes**: For $500 total stake, calculate arbitrage stakes for odds: 2.10, 3.80, 4.20

3. **Value bet**: Your model predicts 60% home win. Odds are 1.80. Calculate EV for $20 bet.

4. **Compare opportunities**: Which is better: 55% at 2.00 odds or 40% at 3.00 odds?

5. **Scan matches**: Create a function to scan 20 matches and rank by EV

6. **Simulate betting**: Simulate 1000 bets with 52% edge. What's probability of profit?

7. **Minimum edge**: What minimum edge do you need to overcome a 5% overround?

8. **Multi-bookmaker**: Compare 3 bookmakers and find best arbitrage opportunity